# 06 · Explainable AI: which input matters

Both methods use the **held-out** outer test folds of the main runs and the models saved in notebook 03 (each fold's model explains only its own test boreholes).

| Method | Models | What it measures |
|---|---|---|
| Permutation importance | all | drop in balanced accuracy when one input group is shuffled (the same row order for every column of the group). Groups: Easting, Northing, Elevation, **Location** (E + N); with VA-HDW also **d**, **Φ**, **S** (all reference boreholes together) and **VA-HDW all** |
| SHAP (exact Shapley values) | XYZ models only | how much each of the 3 inputs raises or lowers P(Clay) for each cell. With VA-HDW there are 3 + 3 × (number of boreholes) inputs, too many for exact SHAP, so the permutation groups are used |
| Elevation dependence | XYZ models | SHAP of elevation against elevation, next to the observed clay share: a geotechnical plausibility check |

Shuffling Easting and Northing does not change the VA-HDW columns (they were computed from the true location); the group **VA-HDW all** measures the information the reference boreholes add.

**Run time:** see the README. Set `XAI_RUNS` to fewer runs to make it faster.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
import joblib
import shap
from sklearn.metrics import balanced_accuracy_score

N_REPEATS = 5                            # shuffles per input group and fold
SHAP_CELLS_PER_FOLD = S["quick"]["shap_cells"] if S["run"]["quick"] else 200
BACKGROUND = 100                         # training cells used as the SHAP background

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
folds, block_m = H.load_folds(S, 1)
preds = pd.read_csv(H.out_dir(S, "03_spatial_cv", private=True) / "outer_predictions.csv")
preds = preds[(preds.multiple == 1) & (preds.purge_x == 0)]
XAI_RUNS = sorted(set(zip(preds.model, preds.features)))     # e.g. [("XGB", "XYZ"), ("XGB", "XYZ_VAHDW")] for a faster run
MODELS = H.out_dir(S, "03_spatial_cv", private=True) / "models"
OUT, PRIV = H.out_dir(S, "06_xai"), H.out_dir(S, "06_xai", private=True)
SHORT = {"UTM_Easting_m": "Easting", "UTM_Northing_m": "Northing", "Top_Elevation_m": "Elevation"}
print("Runs explained:", XAI_RUNS)

def groups_of(columns):
    idx = {c: i for i, c in enumerate(columns)}
    g = {"Easting": [0], "Northing": [1], "Elevation": [2], "Location (E+N)": [0, 1]}
    if len(columns) > 3:
        for kind, label in (("d", "VA-HDW d"), ("phi", "VA-HDW Φ"), ("s", "VA-HDW S")):
            g[label] = [i for c, i in idx.items() if c.startswith(kind + "_")]
        g["VA-HDW all"] = list(range(3, len(columns)))
    return g

## 1. Permutation importance (held-out)

In [ ]:
rng = np.random.default_rng(S["run"]["seed"])
rows = []
for name, features in XAI_RUNS:
    for k, test in preds[(preds.model == name) & (preds.features == features)].groupby("outer_fold"):
        model = joblib.load(MODELS / f"{name}_{features}_x1_purge0_fold{k}.joblib")
        thr = test.threshold.iloc[0]
        for subset, t in (("all", test), ("measured", test[test.measured == 1])):
            X = model.matrix(t)
            base = balanced_accuracy_score(t.y, (model.p_from_matrix(X) >= thr).astype(int))
            for label, cols in groups_of(model.columns()).items():
                drops = []
                for _ in range(N_REPEATS):
                    Xs = X.copy()
                    Xs[:, cols] = X[rng.permutation(len(X))][:, cols]       # one permutation for the whole group
                    drops.append(base - balanced_accuracy_score(t.y, (model.p_from_matrix(Xs) >= thr).astype(int)))
                rows.append({"model": name, "features": features, "outer_fold": k, "cells": subset, "input": label,
                             "kind": "group" if len(cols) > 1 else "single", "drop": float(np.mean(drops))})
    print(name, features, "done", flush=True)
perm = pd.DataFrame(rows)
perm_sum = perm.groupby(["model", "features", "cells", "kind", "input"]).drop.agg(["mean", "std"]).reset_index()
perm_sum.to_csv(OUT / "permutation_importance.csv", index=False)
display(perm_sum[perm_sum.cells == "all"].pivot_table(index=["model", "features"], columns="input", values="mean").round(3))

## 2. SHAP values (held-out cells, XYZ models)

In [ ]:
shap_rows = []
for name, features in [r for r in XAI_RUNS if r[1] == "XYZ"]:
    for k, test in preds[(preds.model == name) & (preds.features == features)].groupby("outer_fold"):
        model = joblib.load(MODELS / f"{name}_{features}_x1_purge0_fold{k}.joblib")
        train, _ = H.outer_split(cells, folds, k)
        bg = train.sample(min(BACKGROUND, len(train)), random_state=k)[H.FEATURES].values
        ex = test.sample(min(SHAP_CELLS_PER_FOLD, len(test)), random_state=k)
        sv = shap.explainers.Exact(model.p_from_matrix, shap.maskers.Independent(bg, max_samples=BACKGROUND))(
            ex[H.FEATURES].values, silent=True)
        out = ex[["BH_Code", "cell", "depth_top_m", "measured", "y", *H.FEATURES]].copy()
        for j, c in enumerate(H.FEATURES):
            out[f"shap_{SHORT[c]}"] = sv.values[:, j]
        shap_rows.append(out.assign(model=name, features=features, outer_fold=k, base=sv.base_values))
    print(name, features, "done", flush=True)
shap_df = pd.concat(shap_rows, ignore_index=True)
shap_df.to_csv(PRIV / "shap_values_by_cell.csv", index=False)
mean_abs = shap_df.groupby("model")[[f"shap_{v}" for v in SHORT.values()]].apply(lambda d: d.abs().mean())
mean_abs.to_csv(OUT / "shap_mean_abs_XYZ.csv")
display(mean_abs.round(4))

## 3. Figures (quick look; the thesis figures are in notebook 09)

In [ ]:
p = perm_sum[perm_sum.cells == "all"].pivot_table(index=["model", "features"], columns="input", values="mean")
cols = [c for c in ["Location (E+N)", "Elevation", "VA-HDW d", "VA-HDW Φ", "VA-HDW S", "VA-HDW all"] if c in p]
fig, ax = plt.subplots(figsize=(11, 4))
p[cols].plot.bar(ax=ax, width=0.85)
ax.set_ylabel("Drop in balanced accuracy"); ax.set_xlabel(""); ax.legend(frameon=False, fontsize=8, ncol=3)
ax.set_title("Held-out permutation importance (all test cells)", fontsize=10)
fig.savefig(OUT / "fig_permutation_importance.png", dpi=200, bbox_inches="tight")

In [ ]:
xyz_models = sorted(set(shap_df.model))
fig, axes = plt.subplots(1, len(xyz_models) + 1, figsize=(3.2 * (len(xyz_models) + 1), 4), sharey=True)
bins = pd.cut(cells.Top_Elevation_m, 20)
prof = cells.groupby(bins, observed=True).agg(z=("Top_Elevation_m", "mean"), clay=("y", "mean"))
axes[0].plot(prof.clay, prof.z, "-o", color="black", markersize=3)
axes[0].set_xlabel("Observed clay share"); axes[0].set_ylabel("Top elevation of cell (m)")
for ax, name in zip(axes[1:], xyz_models):
    d = shap_df[shap_df.model == name]
    ax.scatter(d.shap_Elevation, d.Top_Elevation_m, s=4, alpha=0.4, color="#A0004E")
    ax.axvline(0, color="grey", linewidth=0.8)
    ax.set_xlabel("SHAP of elevation"); ax.set_title(f"{name} | XYZ")
fig.suptitle("Elevation: observed clay share vs the models' elevation effect")
fig.tight_layout(); fig.savefig(OUT / "fig_elevation_dependence.png", dpi=200, bbox_inches="tight")
print(f"Finished in {(time.time() - T0) / 60:.1f} min. Next: notebook 07.")